# 01 — Exploración (Data Understanding)

**Objetivo:** entender los datos demográficos de Naciones Unidas antes de tocarlos: qué significa cada columna, qué representa cada fila, cuántos nulos hay, qué rangos y distribuciones tienen y cómo se enlazan con PISA.

Datasets explorados (World Population Prospects 2024):

- `WPP2024_Demographic_Indicators_Medium_csv.gz`: 54 indicadores demográficos por país/región y año, variante media.
- `WPP2024_Supplement_Births_and_LE60.csv`: nacimientos por sexo y esperanza de vida a los 60 años.
- `WPP2024_Demographic_Indicators_OtherVariants_csv.gz`: los mismos indicadores con otros escenarios de proyección e intervalos de predicción (2024-2101).
- `WPP2024_Demographic_Indicators_notes.csv`: diccionario de los indicadores (nombre, tema y unidad).

- Gráficos rápidos, solo para mí: estoy buscando patrones, no hace falta que sean bonitos.
- Aquí **no se modifica ni se guarda nada**: solo se lee de `data/raw/` (y `data/processed/pisa_unificado.csv` para comprobar el enlace).
- Lo que encuentre lo apunto al final: será la lista de tareas de `02_limpieza`.

### Criterios de trabajo (técnicos y éticos, ASA)

Reglas que sigo en todo el notebook y que se mantienen en los siguientes:

1. **Sin doble conteo:** `Location` mezcla países con agregados. Para comparar o sumar países se filtra siempre `LocTypeName == "Country/Area"` (equivale a `ISO3_code.notna()`, se comprueba en la sección 3).
2. **Unidades explícitas:** los conteos (población, nacimientos, muertes, migrantes) están en **miles**; las tasas, por 1.000 habitantes o nacidos vivos; `TFR`, en hijos por mujer. Para cifras absolutas, multiplicar por 1.000.
3. **Hechos vs. supuestos:** 1950-2023 son **estimaciones** (censos, registros, encuestas y modelos); 2024-2101 son **proyecciones** que dependen de supuestos. Cada resultado indica de qué periodo sale y los gráficos marcan la frontera.
4. **La variante media no es una certeza:** cualquier resultado prospectivo se contrasta con los intervalos de predicción y los escenarios de `OtherVariants`.
5. **Sin selección a conveniencia:** los años de corte se fijan antes de mirar los resultados (ciclos de PISA, frontera 2023) y se documentan; no se eligen variantes ni años para respaldar una narrativa.
6. **Calidad desigual de los datos:** en países con registros civiles incompletos o en conflicto, las estimaciones históricas se apoyan en censos antiguos o modelos indirectos y tienen más error. WPP no publica ese error por país en estos archivos: se tiene en cuenta al interpretar.
7. **Contexto y no estigmatización:** fecundidad, mortalidad infantil y migración se interpretan junto a sus determinantes socioeconómicos y geopolíticos, sin determinismo demográfico ni lecturas que estigmaticen regiones o flujos migratorios.
8. **Reproducibilidad:** todo se carga directamente de los `.csv`/`.csv.gz` originales de `data/raw/`; no hay pasos manuales ni procesos aleatorios.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # permite importar src/

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import DATA_RAW, cargar_raw, cargar_processed, resumen
from src.estilo import aplicar_estilo, CMAP_DIVERGENTE, CMAP_SECUENCIAL, DESTACADO, GRIS

aplicar_estilo()
pd.set_option("display.max_columns", None)

ULTIMO_ESTIMADO = 2023  # último año estimado de WPP2024; desde 2024 todo es proyección

## 1. Carga de datos

**Fuente:** Naciones Unidas, División de Población, [World Population Prospects 2024](https://population.un.org/wpp/). Descargado el 2026-10-14.

- Estimaciones de 1950 a 2023; **proyecciones** (variante media) de 2024 a 2101.
- Cifras absolutas (población, nacimientos, muertes, migrantes) en **miles**.
- Los `.csv.gz` se leen directamente comprimidos (`compression="gzip"`) y con `encoding="utf-8-sig"`, que ignora el BOM si el archivo lo trae.

In [ ]:
# Archivos disponibles en data/raw/
[p.name for p in DATA_RAW.iterdir() if p.name != ".gitkeep"]

In [ ]:
dem = cargar_raw(
    "WPP2024_Demographic_Indicators_Medium_csv.gz",
    compression="gzip",
    encoding="utf-8-sig",
    low_memory=False,
)
print("Medium:", dem.shape)
dem.head()

In [ ]:
sup = cargar_raw("WPP2024_Supplement_Births_and_LE60.csv", encoding="utf-8-sig")
print("Supplement:", sup.shape)
sup.head()

In [ ]:
dic = cargar_raw("WPP2024_Demographic_Indicators_notes.csv", encoding="utf-8-sig")
print("Diccionario:", dic.shape)
dic

## 2. Diccionario de columnas

Los 54 indicadores del Medium están descritos en `dic` (arriba). Las columnas que no son indicadores:

| Columna | Significado | Tipo esperado | Notas |
|---|---|---|---|
| `SortOrder` | Orden de presentación de la ONU | int | Solo en agregados; vacío en países |
| `LocID` | Código numérico de ubicación (M49) | int | Clave estable, también en el Supplement |
| `Notes` | Código de nota al pie | texto | El texto de la nota no viene en los archivos |
| `ISO3_code` / `ISO2_code` | Código ISO del país | texto | Vacío en agregados; **clave para unir con PISA** |
| `SDMX_code` | Código SDMX | int | Mayormente vacío |
| `LocTypeID` / `LocTypeName` | Tipo de ubicación (país, región, grupo de ingreso...) | categórica | Hay que filtrar `Country/Area` |
| `ParentID` | `LocID` de la región que lo contiene | int | Solo en agregados |
| `Location` | Nombre de la ubicación (en inglés) | texto | |
| `VarID` / `Variant` | Variante de proyección | categórica | Siempre `Medium` en este archivo |
| `Time` | Año | int | 1950–2101 |

**Supplement** (ya en el Medium: `LocID`, `ISO3_code`, `Location`, `Time`):

| Columna | Significado | Unidad | Notas |
|---|---|---|---|
| `BirthsMale` / `BirthsFemale` | Nacimientos de niños / niñas | miles | Su suma debería ser `Births` |
| `LE60` | Esperanza de vida a los 60 años, ambos sexos | años | Años que le quedan por vivir a quien llega a 60 |
| `LE60Male` / `LE60Female` | Ídem por sexo | años | |

In [ ]:
# ¿Todas las columnas de indicadores del Medium están en el diccionario?
ids = [
    "SortOrder",
    "LocID",
    "Notes",
    "ISO3_code",
    "ISO2_code",
    "SDMX_code",
    "LocTypeID",
    "LocTypeName",
    "ParentID",
    "Location",
    "VarID",
    "Variant",
    "Time",
]
indicadores = [c for c in dem.columns if c not in ids]
print("Indicadores en el Medium:", len(indicadores))
print("Sin descripción:", set(indicadores) - set(dic["Indicator"]))
print("Descritos pero ausentes:", set(dic["Indicator"]) - set(indicadores))

In [ ]:
dic.groupby("Topic")["Indicator"].apply(list)

In [ ]:
# Unidades: qué indicadores son conteos (en miles) y cuáles son tasas o medidas relativas
dic.groupby("Unit")["Indicator"].apply(list).sort_values(key=lambda s: -s.str.len())

In [ ]:
dem.info()

In [ ]:
sup.info()

## 3. Estructura: qué representa cada fila

En el Medium una fila es **una ubicación en un año**, pero las ubicaciones mezclan países con agregados (Mundo, continentes, grupos de ingreso...). Si se suman o promedian sin filtrar, se cuenta todo varias veces.

In [ ]:
dem["LocTypeName"].value_counts()

In [ ]:
# Variante y rango de años
print(dem["Variant"].value_counts())
print("Años:", dem["Time"].min(), "-", dem["Time"].max())
print("Años por ubicación:", dem.groupby("LocID").size().unique())

In [ ]:
# Agregados disponibles (útiles para comparar contra el promedio mundial o regional)
dem.loc[dem["LocTypeName"] != "Country/Area"].groupby("LocTypeName")[
    "Location"
].unique()

In [ ]:
# ¿Filtrar por LocTypeName o por ISO3_code da lo mismo?
pd.crosstab(dem["LocTypeName"], dem["ISO3_code"].notna().rename("tiene ISO3"))

In [ ]:
# Qué pasa si se suma sin filtrar: población de 2023 (en miles)
d = dem.loc[dem["Time"] == ULTIMO_ESTIMADO]
print(
    f"Suma de todas las filas:   {d['TPopulation1July'].sum() * 1000:>20,.0f} personas"
)
print(
    f"Suma solo de países:       {d.loc[d['ISO3_code'].notna(), 'TPopulation1July'].sum() * 1000:>20,.0f} personas"
)
print(
    f"Fila 'World':              {d.loc[d['LocTypeName'] == 'World', 'TPopulation1July'].iloc[0] * 1000:>20,.0f} personas"
)

In [ ]:
# Me quedo con los países para el resto de la exploración y marco qué filas son proyección
paises = dem.loc[dem["LocTypeName"] == "Country/Area"].copy()
paises["es_proyeccion"] = paises["Time"] > ULTIMO_ESTIMADO
print("Países:", paises["ISO3_code"].nunique(), "| filas:", len(paises))
print("Países sin ISO3:", paises["ISO3_code"].isna().sum())
paises["es_proyeccion"].value_counts()

## 4. Nulos, tipos y duplicados

In [ ]:
resumen(paises)

In [ ]:
# Casi todos los indicadores tienen 237 nulos (= nº de países): ¿un año sin datos?
print("Años con TFR nulo:", paises.loc[paises["TFR"].isna(), "Time"].unique())
paises.loc[paises["Time"] == 2101].dropna(axis=1, how="all").columns.tolist()

In [ ]:
# DoublingTime: ¿cuándo se deja vacío?
validos = paises["Time"].lt(2101)
nulos_dt = paises["DoublingTime"].isna() & validos
print(
    "Crecimiento máx. con DoublingTime nulo:",
    paises.loc[nulos_dt, "PopGrowthRate"].max(),
    "%",
)
print(
    "Crecimiento mín. con DoublingTime informado:",
    paises.loc[~nulos_dt & validos, "PopGrowthRate"].min(),
    "%",
)

In [ ]:
# Notas al pie: qué países las tienen
paises.loc[paises["Notes"].notna(), ["Location", "Notes"]].drop_duplicates()

In [ ]:
resumen(sup)

In [ ]:
print("Duplicados Medium (ISO3, año):", paises.duplicated(["ISO3_code", "Time"]).sum())
print("Duplicados Supplement (ISO3, año):", sup.duplicated(["ISO3_code", "Time"]).sum())
print(
    "Filas duplicadas Medium:",
    dem.duplicated().sum(),
    "| Supplement:",
    sup.duplicated().sum(),
)

## 5. Rangos y distribuciones

Separo estimaciones (1950-2023) de proyecciones (2024-2100): mezclarlas en un mismo resumen confunde datos con supuestos.

In [ ]:
estimados = paises.loc[~paises["es_proyeccion"]]
estimados[indicadores].describe().T

In [ ]:
# Rangos de las proyecciones (sin 2101, que está vacío)
paises.loc[paises["es_proyeccion"] & paises["Time"].lt(2101), indicadores].describe().T

In [ ]:
sup.describe().T

In [ ]:
# Distribución de los indicadores principales en un año de PISA (2022)
principales = [
    "TFR",
    "LEx",
    "IMR",
    "MedianAgePop",
    "CBR",
    "CDR",
    "PopGrowthRate",
    "CNMR",
    "PopDensity",
]
anio = paises.loc[paises["Time"] == 2022]
anio[principales].hist(bins=30, figsize=(14, 10))
plt.tight_layout()

In [ ]:
# Valores extremos en las estimaciones: ¿errores o casos reales?
for col in ["PopDensity", "CNMR", "PopSexRatio", "IMR", "SRB"]:
    print(f"--- {col}")
    display(estimados.nlargest(5, col)[["Location", "Time", col]])

In [ ]:
# Indicadores en los que hay valores negativos (estimaciones)
(estimados[indicadores] < 0).sum()[lambda s: s > 0]

## 6. Evolución temporal

Línea vertical: último año estimado (2023). Lo que está a la derecha son proyecciones, no datos observados.

Aun a la izquierda, no todo es observación directa: en países sin registro civil completo, la ONU estima con censos espaciados, encuestas y modelos indirectos, así que las series históricas de esos países son más inciertas (y más suaves) de lo que parecen.

In [ ]:
mundo = dem.loc[dem["LocTypeName"] == "World"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(
    axes.flat, ["TPopulation1July", "TFR", "LEx", "MedianAgePop", "IMR", "CBR"]
):
    ax.plot(mundo["Time"], mundo[col])
    ax.axvline(ULTIMO_ESTIMADO, color=GRIS, ls="--")
    ax.set_title(col, fontsize=11)
plt.tight_layout()

In [ ]:
# Dispersión entre países a lo largo del tiempo (percentiles 10, 50 y 90 de la fecundidad)
q = paises.groupby("Time")["TFR"].quantile([0.1, 0.5, 0.9]).unstack()
ax = q.plot(figsize=(10, 5), title="TFR entre países: p10, mediana, p90")
ax.axvline(ULTIMO_ESTIMADO, color=GRIS, ls="--")

In [ ]:
# Saltos bruscos de un año al siguiente (shocks: guerras, hambrunas, epidemias, COVID...)
salto = paises.groupby("ISO3_code")["LEx"].diff().abs()
paises.assign(salto_LEx=salto).nlargest(10, "salto_LEx")[
    ["Location", "Time", "LEx", "salto_LEx", "es_proyeccion"]
]

## 7. Supplement: coherencia con el Medium

El Supplement debería ser un complemento del Medium (mismos países y años). Compruebo cobertura, que los nacimientos por sexo sumen el total y que la razón de sexo al nacer implícita cuadre con `SRB`.

In [ ]:
print("Años Supplement:", sup["Time"].min(), "-", sup["Time"].max())
print("Países solo en el Medium:", set(paises["ISO3_code"]) - set(sup["ISO3_code"]))
print("Países solo en el Supplement:", set(sup["ISO3_code"]) - set(paises["ISO3_code"]))

In [ ]:
cruce = paises.merge(
    sup, on=["ISO3_code", "Time"], suffixes=("", "_sup"), validate="one_to_one"
)
print("Filas unidas:", len(cruce))
print("LocID distinto:", (cruce["LocID"] != cruce["LocID_sup"]).sum())
print("Nombre distinto:", (cruce["Location"] != cruce["Location_sup"]).sum())

dif_births = (cruce["BirthsMale"] + cruce["BirthsFemale"] - cruce["Births"]).abs()
print("Máx. |BirthsMale + BirthsFemale - Births| (miles):", dif_births.max().round(4))

srb_impl = 100 * cruce["BirthsMale"] / cruce["BirthsFemale"]
dif_srb = (srb_impl - cruce["SRB"]).abs()
print("Máx. |SRB implícita - SRB|:", dif_srb.max().round(2))
# Las diferencias grandes vienen del redondeo a miles en territorios con muy pocos nacimientos
print(
    "Filas con diferencia > 1:",
    (dif_srb > 1).sum(),
    "| máx. de Births en esas filas (miles):",
    cruce.loc[dif_srb > 1, "Births"].max(),
)

In [ ]:
# LE60 debería estar entre LE65 y LE15 - 45
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(cruce["LE65"], cruce["LE60"], s=3, alpha=0.3)
axes[0].set(xlabel="LE65", ylabel="LE60", title="LE60 vs LE65")
axes[1].scatter(cruce["LEx"], cruce["LE60"], s=3, alpha=0.3)
axes[1].set(
    xlabel="LEx (al nacer)", ylabel="LE60", title="LE60 vs esperanza de vida al nacer"
)
plt.tight_layout()

print("Filas con LE60 <= LE65:", (cruce["LE60"] <= cruce["LE65"]).sum())

In [ ]:
# Brecha por sexo en la esperanza de vida a los 60
brecha = sup.assign(brecha_LE60=sup["LE60Female"] - sup["LE60Male"])
print("Filas con LE60 de hombres > mujeres:", (brecha["brecha_LE60"] < 0).sum())
brecha.groupby("Time")["brecha_LE60"].median().plot(
    title="Mediana entre países: LE60 mujeres - hombres (años)"
)

## 8. Relaciones entre variables

Uso 2022 porque es el último ciclo de PISA con datos estimados (no proyectados), y 1970 solo como referencia histórica. Las correlaciones son descriptivas: no implican causalidad y esconden determinantes comunes (ingreso, educación, acceso a salud).

In [ ]:
# Correlaciones entre indicadores principales (2022, solo países)
vars_corr = principales + ["LE60", "Q5", "Births1519", "NRR", "MAC"]
corr = cruce.loc[cruce["Time"] == 2022, vars_corr].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap=CMAP_DIVERGENTE, vmin=-1, vmax=1, ax=ax)

In [ ]:
# Fecundidad vs esperanza de vida, 1970 vs 2022
fig, ax = plt.subplots(figsize=(9, 6))
for anio_, color in [(1970, GRIS), (2022, DESTACADO)]:
    d = paises.loc[paises["Time"] == anio_]
    ax.scatter(d["TFR"], d["LEx"], s=12, alpha=0.7, color=color, label=anio_)
ax.set(xlabel="TFR (hijos por mujer)", ylabel="LEx (años)")
ax.legend()

In [ ]:
# Muchos indicadores son casi redundantes: ¿cuáles están correlacionados por encima de 0.95?
todos = cruce.loc[cruce["Time"] == 2022, indicadores + ["LE60"]].corr().abs()
pares = todos.where(lambda m: m.gt(0.95)).stack()
pares[[a < b for a, b in pares.index]].sort_values(ascending=False).head(30)

## 9. Otras variantes de proyección (`OtherVariants`)

Mismas columnas que el Medium, pero solo para los años de proyección (2024 en adelante) y con otros supuestos. Hay dos familias:

- **Escenarios deterministas:** cambian un supuesto y mantienen el resto como en el Medium. Por ejemplo, `High`/`Low` (fecundidad ±0,5 hijos), `Constant fertility`, `Zero migration`, `Constant mortality`, `Momentum`, `No change`...
- **Proyección probabilística:** `Median PI` y los intervalos de predicción del 80 % y 95 % (`Lower/Upper 80 PI`, `Lower/Upper 95 PI`), más `Mean`. Sirven para medir la **incertidumbre** de la proyección.

In [ ]:
var = cargar_raw(
    "WPP2024_Demographic_Indicators_OtherVariants_csv.gz",
    compression="gzip",
    encoding="utf-8-sig",
    low_memory=False,
)
print("OtherVariants:", var.shape)
print("Mismas columnas que el Medium:", list(var.columns) == list(dem.columns))
var.head()

In [ ]:
# Variantes: años, filas y nº de ubicaciones de cada una
var.groupby(["VarID", "Variant"]).agg(
    filas=("Time", "size"),
    desde=("Time", "min"),
    hasta=("Time", "max"),
    ubicaciones=("LocID", "nunique"),
)

In [ ]:
var_paises = var.loc[
    (var["LocTypeName"] == "Country/Area") & (var["Time"] < 2101)
].copy()
print(
    "Países por variante:",
    var_paises.groupby("Variant")["ISO3_code"].nunique().unique(),
)
print(
    "Duplicados (ISO3, variante, año):",
    var_paises.duplicated(["ISO3_code", "Variant", "Time"]).sum(),
)
print("Filas duplicadas:", var.duplicated().sum())

In [ ]:
# Cobertura: qué indicadores trae cada variante (proporción de valores no nulos)
cobertura = var_paises.groupby("Variant")[indicadores].apply(lambda d: d.notna().mean())
fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(cobertura, cmap=CMAP_SECUENCIAL, vmin=0, vmax=1, ax=ax)
ax.set_title(
    "Proporción de valores no nulos por variante e indicador (países, 2024-2100)"
)

In [ ]:
# Columnas que trae la variante Mean
cobertura.loc["Mean"][lambda s: s > 0]

In [ ]:
# Comparación con el Medium en los mismos países y años
clave = ["ISO3_code", "Time"]
for v in ["High", "Low", "Median PI", "Constant fertility", "Zero migration"]:
    j = var_paises.loc[var_paises["Variant"] == v].merge(
        paises[clave + ["TFR", "TPopulation1July"]], on=clave, suffixes=("", "_med")
    )
    d_tfr = j["TFR"] - j["TFR_med"]
    d_pop = (j["TPopulation1July"] / j["TPopulation1July_med"] - 1) * 100
    print(
        f"{v:20s} TFR - Medium: [{d_tfr.min():+.2f}, {d_tfr.max():+.2f}]  |  "
        f"población 2100 vs Medium (mediana): {d_pop[j['Time'] == 2100].median():+.1f} %"
    )

In [ ]:
# Población mundial según cada escenario
var_mundo = var.loc[var["LocTypeName"] == "World"]
fig, ax = plt.subplots(figsize=(11, 6))
for v, d in var_mundo.loc[var_mundo["VarID"] < 200].groupby("Variant"):
    ax.plot(d["Time"], d["TPopulation1July"] / 1e6, lw=1, label=v)
med = mundo.loc[mundo["Time"].between(1950, 2100)]
ax.plot(
    med["Time"], med["TPopulation1July"] / 1e6, color="black", lw=2.5, label="Medium"
)
ax.axvline(ULTIMO_ESTIMADO, color=GRIS, ls="--")
ax.set(ylabel="Población (miles de millones)", title="Población mundial por escenario")
ax.legend(fontsize=8, ncol=2)

In [ ]:
# Intervalos de predicción: TFR y esperanza de vida del mundo
pi = var_mundo.pivot_table(index="Time", columns="Variant", values=["TFR", "LEx"])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col in zip(axes, ["TFR", "LEx"]):
    ax.fill_between(
        pi.index,
        pi[(col, "Lower 95 PI")],
        pi[(col, "Upper 95 PI")],
        color=DESTACADO,
        alpha=0.15,
        label="95 %",
    )
    ax.fill_between(
        pi.index,
        pi[(col, "Lower 80 PI")],
        pi[(col, "Upper 80 PI")],
        color=DESTACADO,
        alpha=0.3,
        label="80 %",
    )
    ax.plot(mundo["Time"], mundo[col], color="black", label="Medium")
    ax.axvline(ULTIMO_ESTIMADO, color=GRIS, ls="--")
    ax.set_title(f"{col} mundial con intervalos de predicción", fontsize=11)
    ax.legend()
plt.tight_layout()

In [ ]:
# Sensibilidad de una conclusión prospectiva: ¿en qué año alcanza su máximo la población mundial?
pico = (
    pd.concat([var_mundo, mundo.assign(Variant="Medium")])
    .loc[lambda d: d["Time"].between(2024, 2100) & d["Variant"].ne("Mean")]
    .sort_values("TPopulation1July")
    .groupby("Variant")
    .tail(1)
    .set_index("Variant")[["Time", "TPopulation1July"]]
    .rename(columns={"Time": "anio_pico", "TPopulation1July": "poblacion_pico_miles"})
)
pico["sigue_creciendo_en_2100"] = pico["anio_pico"].eq(2100)
pico.sort_values("anio_pico")

In [ ]:
# Coherencia: Lower 95 <= Lower 80 <= Median <= Upper 80 <= Upper 95
orden = ["Lower 95 PI", "Lower 80 PI", "Median PI", "Upper 80 PI", "Upper 95 PI"]
for col in ["TPopulation1July", "TFR", "LEx"]:
    t = var_paises.pivot_table(index=clave, columns="Variant", values=col)[orden]
    print(
        f"{col}: filas con intervalos desordenados =",
        (t.diff(axis=1).iloc[:, 1:] < 0).any(axis=1).sum(),
    )

# ¿Median PI coincide con el Medium?
j = var_paises.loc[var_paises["Variant"] == "Median PI"].merge(
    paises[clave + ["TPopulation1July"]], on=clave, suffixes=("", "_med")
)
print(
    "Máx. |Median PI - Medium| en población (miles):",
    (j["TPopulation1July"] - j["TPopulation1July_med"]).abs().max(),
)

In [ ]:
# Incertidumbre en 2025 (año de PISA proyectado): ancho del intervalo del 95 % en los países de PISA
pisa_iso = cargar_processed("pisa_unificado.csv")["ISO3_code"].unique()
t = var_paises.loc[
    (var_paises["Time"] == 2025) & var_paises["ISO3_code"].isin(pisa_iso)
]
t = t.pivot_table(
    index="ISO3_code", columns="Variant", values=["TFR", "LEx", "TPopulation1July"]
)
ancho = pd.DataFrame(
    {
        "TFR (hijos)": t[("TFR", "Upper 95 PI")] - t[("TFR", "Lower 95 PI")],
        "LEx (años)": t[("LEx", "Upper 95 PI")] - t[("LEx", "Lower 95 PI")],
        "Población (% del Medium)": 100
        * (
            t[("TPopulation1July", "Upper 95 PI")]
            - t[("TPopulation1July", "Lower 95 PI")]
        )
        / t[("TPopulation1July", "Median PI")],
    }
)
ancho.describe().T

## 10. Enlace con PISA

El objetivo es cruzar estos indicadores con `data/processed/pisa_unificado.csv` por `ISO3_code` y `Time`.

In [ ]:
pisa = cargar_processed("pisa_unificado.csv")
print("Ciclos PISA:", sorted(pisa["Time"].unique()))
print("Países PISA:", pisa["ISO3_code"].nunique())
print(
    "Códigos de PISA que no están en WPP:",
    set(pisa["ISO3_code"]) - set(paises["ISO3_code"]),
)

In [ ]:
# Cobertura: ¿cada fila de PISA encuentra su año en WPP?
enlace = pisa[["ISO3_code", "Time"]].merge(
    cruce[["ISO3_code", "Time", "TFR"]],
    on=["ISO3_code", "Time"],
    how="left",
    indicator=True,
)
enlace["_merge"].value_counts()

In [ ]:
# Ojo: PISA 2025 cae en años de proyección de WPP
pisa["Time"].gt(2023).groupby(pisa["Time"]).sum()[lambda s: s > 0]

## 11. Hallazgos → tareas para `02_limpieza`

- [ ] **Filtrar países:** quedarse con `LocTypeName == "Country/Area"` (237; equivale a `ISO3_code.notna()`). Sumar sin filtrar multiplica la población de 2023 por ~45. Si se usan agregados (Mundo, grupos de ingreso) como referencia, guardarlos aparte.
- [ ] **Quitar el año 2101:** solo trae `TPopulation1Jan`; el resto de columnas son nulas.
- [ ] **Separar estimación y proyección:** añadir una columna `es_proyeccion = Time > 2023`. PISA 2025 se cruzaría con datos proyectados.
- [ ] **Unir Medium + Supplement** por `ISO3_code` y `Time` (uno a uno, coinciden `LocID` y nombre). Vaticano (`VAT`) no está en el Supplement.
- [ ] **Descartar columnas sin uso:** `SortOrder`, `SDMX_code`, `LocTypeID`, `ParentID`, `VarID`, `Variant` (constantes o vacías en países). `BirthsMale + BirthsFemale` reproduce `Births`.
- [ ] **`DoublingTime`:** la ONU lo deja vacío cuando el crecimiento es ≤ 0,5 % anual (no solo cuando la población decrece). No imputar; probablemente no usarlo.
- [ ] **Territorios muy pequeños:** con < 200 nacimientos al año, el redondeo a miles distorsiona cocientes (p. ej. la SRB implícita del Supplement difiere hasta 11 puntos). Valorar un umbral de población mínima.
- [ ] **Unidades:** población, nacimientos, muertes y migrantes están en miles (×1.000 para cifras absolutas). `CBR`, `CDR`, `CNMR` son por 1.000 habitantes; `IMR` y `Q5` por 1.000 nacidos vivos; `TFR` en hijos por mujer. Documentar la unidad en cada eje y tabla.
- [ ] **Valores extremos reales, no errores:** densidades de microestados (Mónaco, Macao...), migración neta en países pequeños o en crisis, saltos de `LEx` por guerras o epidemias. Revisarlos al graficar, no eliminarlos.
- [ ] **Indicadores redundantes:** muchas variables están correlacionadas > 0.95 (versiones por sexo, LE15/LE60/LE65, CBR/TFR, IMR/Q5). Elegir pocas para el análisis.
- [ ] **LE60 por sexo:** en 107 filas los hombres superan a las mujeres; comprobar qué países y años antes de usar la brecha.
- [ ] **Enlace con PISA:** 590 de 593 filas de PISA encuentran su país-año en WPP. Las 3 que no son `OED` (promedio OCDE), que no existe en WPP: calcularlo como promedio de los países OCDE o excluirlo.
- [ ] **PISA 2025** (92 países) se cruza con proyecciones de WPP, no con estimaciones: marcarlo o avisarlo en el informe.
- [ ] **OtherVariants solo si hace falta:** son ~663 000 filas (18 variantes × 555 ubicaciones × 2024-2101). Para el análisis principal basta el Medium; usar `Lower/Upper 80/95 PI` solo para mostrar incertidumbre o escenarios (`High`/`Low`) como sensibilidad.
- [ ] **Cobertura desigual por variante:** las variantes probabilísticas (`PI`) no traen `NRR`, `SRB`, muertes por sexo ni muertes infantiles; varios escenarios deterministas no traen `LE15/65/80` ni las probabilidades `Q`. `Mean` solo trae `TFR`, `LExMale` y `LExFemale`. Comprobar la cobertura antes de elegir un indicador.
- [ ] **`Median PI` = Medium:** no aporta datos nuevos; descartarla.
- [ ] **Incertidumbre en PISA 2025:** el intervalo del 95 % de 2025 tiene una mediana de ±0,27 hijos en `TFR` y ±1,5 años en `LEx` en los países de PISA. Si se usan datos de 2025, mencionarlo.
- [ ] **Conclusiones prospectivas, siempre con su rango:** el año pico de la población mundial es 2084 en el Medium, pero va de 2068 (`Lower 95 PI`) a "sigue creciendo en 2100" (`Upper 80/95 PI`). Ningún resultado posterior a 2023 se presenta sin su intervalo o escenario alternativo.
- [ ] _..._

## 12. Criterios para `02_limpieza` y el informe (ASA)

- **Documentar el pipeline:** en `02_limpieza` dejar escritos los criterios de inclusión/exclusión de países (agregados, `OED`, territorios muy pequeños, Vaticano), el tratamiento de nulos (`DoublingTime`, 2101) y cuántas filas afecta cada decisión.
- **Decidir antes de mirar:** los años de análisis son los ciclos de PISA y la frontera es 2023. Si se cambia un criterio, se explica por qué, no porque "salga mejor".
- **Hechos vs. supuestos en el informe:** cada hallazgo indica si se basa en estimaciones (1950-2023) o en proyecciones (2024+), y en este caso con qué variante y qué incertidumbre.
- **Calidad desigual:** mencionar en *Limitaciones* que las estimaciones de países con registros civiles incompletos o en conflicto tienen más error, y que estos archivos no lo cuantifican por país.
- **Contexto y lenguaje:** al hablar de fecundidad, mortalidad infantil o migración, acompañar los datos de sus determinantes (ingreso, educación, conflictos, políticas) y evitar lecturas deterministas o que estigmaticen regiones o flujos migratorios.
- **Reproducibilidad:** cargar siempre desde `data/raw/` con `src/utils.py`; si en algún momento se simula o se muestrea, fijar la semilla.